# Sonifying the Heart

### Inner emotion detection from ECG through music

**ECG data → Heartbeats → Music → Emotion**

A heart does not beat like a metronome. The gap between one beat and the next shifts constantly,
and how it shifts says something about the state of the body producing it. This notebook turns
that shifting into music, and then asks what emotion the music carries.

Everything is in this one file — there is nothing to install from the project and nothing to
import from it. Run the cells from top to bottom.

---

## What the pipeline does

| Stage | What happens |
|---|---|
| **1. ECG data** | Real recordings from PhysioNet, streamed on demand. Any WFDB or CSV file also works. |
| **2. Heartbeats** | Find the R-peaks, from annotations or the XQRS detector. Compute R–R intervals and RMSSD. Pick a 30-second clip. |
| **3. Music** | One note per heartbeat, plus a string harmony and a lub-dub pulse. Render to MIDI, audio and a score. |
| **4. Emotion** | Place the music on Russell's valence–arousal plane. |

## The sonification rule, which is the heart of the project

- Each heartbeat becomes **exactly one note**, starting at the R-peak.
- The note lasts **85% of the gap** to the next beat, so the rhythm of the music *is* the rhythm
  of the heart.
- Its **loudness** comes from the ECG amplitude at that beat.
- Its **pitch** follows how far that beat's interval deviates from the mean of the previous five.
  On time means C4; a faster beat goes **up** the scale, a slower one goes **down**.
- While the deviation stays under **12%** the melody uses the C major pentatonic scale, so it
  stays consonant. Past 12% it goes **chromatic**, and starts to sound wrong.
- The harmony sours in step: a chord turns **diminished** at 12% average deviation, and into a
  **semitone cluster** at 25%.

Nothing in that rule knows what disease the person has. It only knows timing.

## The experiment

Most recordings in the MIT-BIH Atrial Fibrillation Database contain **both** normal rhythm and
atrial fibrillation, so we can hear *the same person* in both states — holding age, physiology,
electrodes and equipment constant. That is the experiment: 7 recordings from 6 people, one normal
clip and one AFib clip each, 14 pieces.

---

## 1. Setup

Install the dependencies if you do not have them:

```
pip install numpy scipy wfdb pretty_midi matplotlib pandas
```

For audio you also need **FluidSynth** and a General MIDI soundfont. On Ubuntu:

```
sudo apt install fluidsynth fluid-soundfont-gm
```

MIDI files and all the plots work without it — only the WAV rendering needs it.

In [ ]:
import csv
import json
import math
import os
import random
import shutil
import statistics
import subprocess
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pretty_midi
import wfdb
from matplotlib.lines import Line2D
from scipy import stats
from scipy.io import wavfile

print("numpy", np.__version__, "| wfdb", wfdb.__version__, "| pandas", pd.__version__)

### Switches

The expensive steps are off by default, so the notebook runs quickly the first time. Turn them
on when you want the full result.

- `RENDER_AUDIO` — render WAV files with FluidSynth. Needs FluidSynth installed.
- `BUILD_ALL` — compose all 14 example pieces (streams 7 records from PhysioNet; a few minutes).
- `BUILD_STUDY` — build the blinded listening-study clip set. Needs `ffmpeg` and `RENDER_AUDIO`.

In [ ]:
RENDER_AUDIO = False
BUILD_ALL    = False
BUILD_STUDY  = False

OUTPUT_DIR   = Path("output_nb")
EXAMPLES_DIR = OUTPUT_DIR / "examples"
STUDY_DIR    = OUTPUT_DIR / "listening_study"

if RENDER_AUDIO and shutil.which("fluidsynth") is None:
    print("FluidSynth not found - turning RENDER_AUDIO off. MIDI and plots still work.")
    RENDER_AUDIO = False

print(f"RENDER_AUDIO={RENDER_AUDIO}  BUILD_ALL={BUILD_ALL}  BUILD_STUDY={BUILD_STUDY}")

---

## 2. Every parameter in one place

These are all the numbers the pipeline uses. They are collected here so that nothing is buried
in a function, and so that the effect of changing one is easy to trace.

A note on how they were chosen: they were calibrated on the 14 example clips, and most have not
yet been tuned by ear. That is listed as future work — for a music project it matters that a
parameter can be justified by how it *sounds*, not only by what it measures.

In [ ]:
# --- ECG data ---------------------------------------------------------------------------------
EXAMPLE_RECORDS = (("afdb", "04015"), ("afdb", "04043"), ("afdb", "04126"), ("afdb", "06995"),
                   ("afdb", "08455"), ("mitdb", "201"), ("mitdb", "202"))
CLIP_SECONDS = 30
CANDIDATE_WINDOWS = 5
MIN_INTERVALS_PER_WINDOW = 5
# afdb's beat marks come from an automatic detector and were never corrected by hand, so they can
# jitter inside a correctly labelled "N" stretch. Keeping the STEADIEST normal window avoids that;
# real sinus rhythm is genuinely low-variability. For AFib we keep the best-detected window.
SELECTION = {"N": "steadiest", "AFIB": "most_beats"}
BEAT_SYMBOLS = frozenset("NLRBAJSVrFejnE/fQ")
MIN_ANALYSIS_SECONDS = 5
MIN_DETECTED_BEATS = 6

# --- Melody -----------------------------------------------------------------------------------
ROOT_NOTE = 60                 # C4: the pitch of a beat that arrives exactly on time
PENTATONIC = (0, 2, 4, 7, 9)   # C D E G A - no semitones, so it cannot sound dissonant
CHROMATIC = tuple(range(12))
OCTAVE_RANGE = 2               # how far an extreme deviation can travel
RUNNING_MEAN_BEATS = 5         # "on time" means: against the mean of the last five intervals
CHROMATIC_THRESHOLD = 0.12     # past 12% deviation the melody leaves the scale
FULL_RANGE_DEVIATION = 0.5     # 50% deviation reaches the top or bottom of the range
NOTE_LENGTH = 0.85             # fraction of the R-R interval
VELOCITY_RANGE = (40, 110)
PITCH_RANGE = (21, 108)
PEAK_SEARCH_SAMPLES = 5

# --- Emotion ----------------------------------------------------------------------------------
QUADRANTS = {
    "happy_excited": {"valence": "+", "arousal": "+", "program": 12, "instrument": "Marimba"},
    "tense_anxious": {"valence": "-", "arousal": "+", "program": 30, "instrument": "Distortion Guitar"},
    "sad_depressed": {"valence": "-", "arousal": "-", "program": 42, "instrument": "Cello"},
    "calm_relaxed":  {"valence": "+", "arousal": "-", "program": 24, "instrument": "Acoustic Guitar (nylon)"},
}
# Tempo and loudness are left out on purpose: tempo varies more between people than between
# rhythms, and loudness is re-scaled inside every clip anyway.
IRREGULARITY_RANGE = (0.0, 0.30)
SPREAD_RANGE = (0.0, 11.0)
CONSONANCE_RANGE = (0.30, 1.0)
AROUSAL_WEIGHTS = (0.55, 0.45)   # rhythmic irregularity, pitch spread
VALENCE_WEIGHTS = (0.70, 0.30)   # consonance, pitch steadiness

# --- Arrangement ------------------------------------------------------------------------------
PROGRESSION = ((48, (0, 4, 7)), (45, (0, 3, 7)), (41, (0, 4, 7)), (43, (0, 4, 7)))  # C Am F G
BEATS_PER_CHORD = 4
TENSE_AT, CHAOTIC_AT = 0.12, 0.25
CHORD_SHAPES = {"tense": (0, 3, 6), "chaotic": (0, 1, 6)}  # diminished triad, semitone cluster
CHORD_VELOCITY = {"consonant": 34, "tense": 44, "chaotic": 54}
PIANO_PROGRAM, STRINGS_PROGRAM, DRUM_KIT_PROGRAM = 0, 48, 0
LUB_DRUM, DUB_DRUM = 35, 36
DRUM_HIT_SECONDS = 0.1
DUB_DELAY_FRACTION, DUB_DELAY_MAX = 0.35, 0.30
DUB_VELOCITY_RATIO = 0.6
PULSE_VELOCITY_RANGE = (30, 70)

# --- Audio ------------------------------------------------------------------------------------
SOUNDFONT = os.environ.get("ECGMUSIC_SOUNDFONT", "/usr/share/sounds/sf2/FluidR3_GM.sf2")
SAMPLE_RATE = 44100
PEAK_LEVEL = 0.85   # raw renders of these sparse parts peak near 5% of full scale

print("parameters loaded")

---

## 3. Stages 1 and 2 — from an ECG recording to heartbeats

### Where the data comes from

| | MIT-BIH Atrial Fibrillation (`afdb`) | MIT-BIH Arrhythmia (`mitdb`) |
|---|---|---|
| Content | 25 recordings of ~10 hours; 23 include the signal | 48 half-hour excerpts |
| People | adults with mostly paroxysmal AFib | 47 people, 25 men aged 32–89 and 22 women aged 23–89 |
| Signal | 2 leads, 250 Hz, 12-bit | 2 leads, 360 Hz, 11-bit |
| Beat marks | `.qrs`, automatic, **not** corrected by hand | inside `.atr`, checked by two or more cardiologists |
| Licence | Open Data Commons Attribution 1.0 | Open Data Commons Attribution 1.0 |

Both are cardiology archives from Beth Israel Hospital, Boston, recorded between 1975 and 1983.
**Neither contains any emotional or psychological information** — which is exactly why Stage 4
cannot be validated on them. More on that at the end.

Records are **streamed** from PhysioNet, so nothing needs downloading: only the annotations and
the 30-second windows actually used are fetched, a few hundred kilobytes per record.

### The `Clip` — an ECG stretch plus the position of every heartbeat

Two numbers matter:

- **mean R–R** — the average gap between beats, which is just heart rate in another form.
- **RMSSD** — the root mean square of successive differences. It measures how much each beat's
  timing differs from the one before, so it is a direct measure of *irregularity*. This is the
  number that separates the two rhythms.

In [ ]:
@dataclass
class Clip:
    """A stretch of ECG together with the sample positions of its heartbeats (R-peaks)."""
    signal: np.ndarray
    peaks: np.ndarray
    fs: float
    label: str = ""

    @property
    def rr(self):
        """Seconds between consecutive heartbeats."""
        return np.diff(self.peaks) / self.fs

    @property
    def rmssd_ms(self):
        diffs = np.diff(self.rr)
        return float(np.sqrt(np.mean(diffs ** 2))) * 1000 if len(diffs) else float("nan")

    @property
    def mean_rr_ms(self):
        rr = self.rr
        return float(np.mean(rr)) * 1000 if len(rr) else float("nan")

    @property
    def bpm(self):
        return 60000 / self.mean_rr_ms if self.mean_rr_ms else float("nan")

### Reading the annotations

The two databases store their labels differently, so each needs its own branch.

- **`afdb`** keeps beats in a `.qrs` file and rhythm labels in `.atr`.
- **`mitdb`** puts everything in `.atr`: beats are rows whose symbol is a beat code, and rhythm
  changes are rows whose symbol is `+`.

In [ ]:
def _rhythm_intervals(samples, notes, total_samples):
    ends = list(samples[1:]) + [total_samples]
    return [(int(start), int(end), note.strip("(").strip("\x00").strip())
            for start, end, note in zip(samples, ends, notes)]


def load_annotations(record, db):
    """Heartbeat sample positions, rhythm intervals (start, end, label) and sampling rate."""
    header = wfdb.rdheader(record, pn_dir=db)
    if db == "afdb":
        beats = wfdb.rdann(record, "qrs", pn_dir=db).sample
        atr = wfdb.rdann(record, "atr", pn_dir=db)
        rhythm_samples, rhythm_notes = atr.sample, atr.aux_note
    elif db == "mitdb":
        atr = wfdb.rdann(record, "atr", pn_dir=db)
        beats = atr.sample[np.array([s in BEAT_SYMBOLS for s in atr.symbol])]
        is_rhythm = np.array([s == "+" for s in atr.symbol])
        rhythm_samples = atr.sample[is_rhythm]
        rhythm_notes = [n for n, keep in zip(atr.aux_note, is_rhythm) if keep]
    else:
        raise ValueError(f"unknown database {db!r} (expected 'afdb' or 'mitdb')")
    return beats, _rhythm_intervals(rhythm_samples, rhythm_notes, header.sig_len), header.fs

### Choosing the 30-second clips — a problem the data forced on us

The obvious thing is to take the first 30 seconds of each labelled stretch. **That does not
work.** Running five records this way made two "normal" clips *more* irregular than their AFib
counterparts — record 04015 gave 199 ms against 105 ms.

The cause is `afdb`'s beat marks: they come from an automatic detector that was never corrected
by hand, so they jitter inside stretches that are correctly labelled normal.

**The fix:** search several candidate windows and keep the **steadiest** normal window — real
sinus rhythm genuinely is low-variability, so the steadiest window is the most honest one — and
for AFib keep the window with the **most detected beats**. After this change the normal clips ran
7.6–18.6 ms against AFib 99.9–141.9 ms.

This is the sort of thing that only shows up when you actually look at the data.

In [ ]:
def _rmssd(rr):
    return float(np.sqrt(np.mean(np.diff(rr) ** 2)))


def choose_window(intervals, label, beats, fs):
    """(start, end) sample of the best CLIP_SECONDS window inside stretches labelled `label`."""
    needed = int(CLIP_SECONDS * fs)
    rule = SELECTION[label]
    best = None
    for start, end, lbl in intervals:
        if lbl != label or end - start < needed:
            continue
        span = end - start - needed
        count = min(CANDIDATE_WINDOWS, span // needed + 1)
        offsets = np.linspace(0, span, count).astype(int) if span > 0 else [0]
        for offset in offsets:
            low = start + int(offset)
            peaks = beats[(beats >= low) & (beats < low + needed)]
            if len(peaks) - 1 < MIN_INTERVALS_PER_WINDOW:
                continue
            score = _rmssd(np.diff(peaks) / fs) if rule == "steadiest" else -len(peaks)
            if best is None or score < best[0]:
                best = (score, low)
    return None if best is None else (best[1], best[1] + needed)


def record_clips(record, db):
    """The chosen normal ("N") and atrial fibrillation ("AFIB") clips of a record, by label."""
    beats, intervals, fs = load_annotations(record, db)
    clips = {}
    for label in SELECTION:
        window = choose_window(intervals, label, beats, fs)
        if window is None:
            continue
        low, high = window
        signal = wfdb.rdrecord(record, sampfrom=low, sampto=high, pn_dir=db).p_signal[:, 0]
        clips[label] = Clip(signal, beats[(beats >= low) & (beats < high)] - low, float(fs), label)
    return clips

### Any ECG file, with no annotations at all

The pipeline does not depend on PhysioNet. `detect_beats` runs the **XQRS** detector, so a plain
recording works. We checked it on record 04043 with its annotations hidden: in the first 30
seconds it found 53 beats with a mean R–R of 559 ms and an RMSSD of 14.2 ms, against 54 beats,
559 ms and 13.8 ms from the annotations.

In [ ]:
def detect_beats(signal, fs):
    """R-peak sample positions found by the XQRS detector - no annotation file needed."""
    from wfdb.processing import xqrs_detect
    return xqrs_detect(signal, fs=fs, verbose=False)


def load_signal(path, fs=None, channel=0):
    """(signal, sampling rate) from a WFDB record (path without extension) or a sample file."""
    if Path(f"{path}.hea").exists():
        record = wfdb.rdrecord(str(path))
        return record.p_signal[:, channel], fs or record.fs
    if not Path(path).exists():
        raise ValueError(f"{path} not found (for a WFDB record, give the path without extension)")
    if fs is None:
        raise ValueError(f"{path} is not a WFDB record, so the sampling rate (fs) is required")
    signal = np.loadtxt(path, delimiter="," if str(path).endswith(".csv") else None)
    return (signal[:, channel] if signal.ndim > 1 else signal), fs


def clip_from_signal(signal, fs, start=0.0, duration=60.0):
    """Cut a window out of a raw ECG signal and find its heartbeats."""
    low = int(start * fs)
    high = min(len(signal), low + int(duration * fs))
    if high - low < MIN_ANALYSIS_SECONDS * fs:
        raise ValueError(f"only {max(high - low, 0) / fs:.1f}s of signal; "
                         f"need at least {MIN_ANALYSIS_SECONDS}s")
    window = np.asarray(signal[low:high])
    peaks = detect_beats(window, fs)
    if len(peaks) < MIN_DETECTED_BEATS:
        raise ValueError(f"only {len(peaks)} heartbeats found; try a longer duration")
    return Clip(window, np.asarray(peaks), float(fs))

### Let's look at one person

Record 04043 contains both rhythms, so we hear the same heart in two states. This streams a few
hundred kilobytes from PhysioNet.

In [ ]:
clips = record_clips("04043", "afdb")

for label, clip in clips.items():
    print(f"{label:>5}: {len(clip.peaks):>3} beats   "
          f"mean R-R {clip.mean_rr_ms:>6.0f} ms ({clip.bpm:>5.1f} bpm)   "
          f"RMSSD {clip.rmssd_ms:>6.1f} ms")

ratio = clips["AFIB"].rmssd_ms / clips["N"].rmssd_ms
print(f"\nThe AFib clip is {ratio:.0f}x as irregular as the normal one.")

In [ ]:
def plot_clip(clip, title="", ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(12, 2.8))
    ax.plot(np.arange(len(clip.signal)) / clip.fs, clip.signal, linewidth=0.6)
    ax.plot(clip.peaks / clip.fs, clip.signal[clip.peaks], "rx", markersize=4,
            label="heartbeat (R-peak)")
    ax.set(title=title or f"{clip.label}: RMSSD {clip.rmssd_ms:.1f} ms, "
                          f"mean R-R {clip.mean_rr_ms:.0f} ms",
           xlabel="Time (s)", ylabel="ECG (mV)")
    ax.legend(loc="upper right", fontsize=8)
    return ax


fig, axes = plt.subplots(2, 1, figsize=(13, 6))
for ax, (label, clip) in zip(axes, clips.items()):
    name = "Normal rhythm" if label == "N" else "Atrial fibrillation"
    plot_clip(clip, f"04043 - {name}: RMSSD {clip.rmssd_ms:.1f} ms", ax)
fig.tight_layout()
plt.show()

The difference is already visible. In the normal clip the red crosses are evenly spaced. In the
AFib clip the gaps between them vary visibly from beat to beat — and that variation is the only
thing the music is built from.

---

## 4. Stage 3a — the melody: one note per heartbeat

This is the core of the project, so it is worth being precise about what each beat becomes.

For beat *i*, let $r_i$ be the interval to the next beat and $\bar{r}_i$ the mean of the last
five intervals. The **deviation** is

$$d_i = \frac{r_i - \bar{r}_i}{\bar{r}_i}$$

so $d_i = 0$ means the beat arrived exactly on schedule, $d_i < 0$ means it came **early** (a
faster beat), and $d_i > 0$ means it came **late**.

| Property of the note | Comes from |
|---|---|
| **start** | the R-peak itself |
| **duration** | 85% of $r_i$ |
| **loudness** | the ECG amplitude at that beat |
| **pitch** | $d_i$ — early goes up, late goes down |
| **scale** | pentatonic while $\lvert d_i \rvert < 12\%$, chromatic beyond |

Using a **running** mean rather than the clip average matters: it makes the music respond to
*changes* in rhythm rather than to a person's baseline heart rate. A steady fast heart still
plays a steady line.

Why the pentatonic scale: it contains no semitones, so no two notes in it can clash. A steady
heart is therefore *incapable* of sounding dissonant — the dissonance in these pieces only ever
comes from genuine irregularity, never from the choice of notes.

In [ ]:
@dataclass
class Note:
    pitch: int
    start: float
    duration: float
    velocity: int
    deviation: float = 0.0
    rr: float = 0.0
    chromatic: bool = False


def _scale_below(scale):
    """The same scale walked downward from the root: C D E G A -> C A G E D."""
    return sorted((12 - step) % 12 for step in scale)


def _offset(step, scale):
    octave, degree = divmod(step, len(scale))
    return octave * 12 + scale[degree]


def deviation_to_pitch(deviation):
    """MIDI pitch for a beat's R-R deviation: faster beats go up, slower ones down the scale."""
    scale = PENTATONIC if abs(deviation) < CHROMATIC_THRESHOLD else CHROMATIC
    steps = len(scale) * OCTAVE_RANGE
    step = round(float(min(abs(deviation) / FULL_RANGE_DEVIATION, 1.0)) * steps)
    if deviation < 0:
        pitch = ROOT_NOTE + _offset(step, scale)
    else:
        pitch = ROOT_NOTE - _offset(step, _scale_below(scale))
    low, high = PITCH_RANGE
    return int(min(max(pitch, low), high))


def running_means(rr):
    """Mean of each interval and up to RUNNING_MEAN_BEATS - 1 intervals before it."""
    n = RUNNING_MEAN_BEATS
    return np.array([np.mean(rr[max(0, i - n + 1):i + 1]) for i in range(len(rr))])


def beat_velocities(signal, peaks):
    """Note loudness from the ECG amplitude at each beat, scaled per clip."""
    # automatic R-peak marks can land a few samples off the apex, so take the local maximum
    w = PEAK_SEARCH_SAMPLES
    amplitudes = np.array([signal[max(0, p - w):p + w + 1].max() for p in peaks])
    low, high = VELOCITY_RANGE
    if amplitudes.max() <= amplitudes.min():
        return np.full(len(peaks), (low + high) // 2)
    scaled = np.interp(amplitudes, [amplitudes.min(), amplitudes.max()], [low, high])
    return np.clip(scaled, low, high).astype(int)


def melody(clip):
    """One note per heartbeat (the last beat has no following interval, so it is dropped)."""
    rr = clip.rr
    if len(rr) == 0:
        return []
    means = running_means(rr)
    velocities = beat_velocities(np.asarray(clip.signal), clip.peaks)
    notes = []
    for i, interval in enumerate(rr):
        deviation = float((interval - means[i]) / means[i])
        notes.append(Note(pitch=deviation_to_pitch(deviation),
                          start=float(clip.peaks[i] / clip.fs),
                          duration=float(interval * NOTE_LENGTH),
                          velocity=int(velocities[i]),
                          deviation=deviation,
                          rr=float(interval),
                          chromatic=abs(deviation) >= CHROMATIC_THRESHOLD))
    return notes

### What the rule does, before any heartbeat is involved

A quick sanity check: feed the rule a range of deviations and see which pitch comes out.

In [ ]:
NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
note_name = lambda p: f"{NAMES[p % 12]}{p // 12 - 1}"

rows = []
for d in (-0.50, -0.30, -0.15, -0.12, -0.05, 0.0, 0.05, 0.12, 0.15, 0.30, 0.50):
    pitch = deviation_to_pitch(d)
    rows.append({"deviation": f"{d:+.0%}",
                 "beat arrived": "on time" if d == 0 else ("early (faster)" if d < 0 else "late (slower)"),
                 "pitch": note_name(pitch),
                 "scale": "pentatonic" if abs(d) < CHROMATIC_THRESHOLD else "chromatic"})
pd.DataFrame(rows)

Note what happens at exactly 12%: the scale switches, and the melody starts using notes that
clash. That threshold is the single most important number in the project — it is the point at
which an irregular heart becomes *audible* as wrongness rather than just as movement.

### The melody for our two clips

In [ ]:
melodies = {label: melody(clip) for label, clip in clips.items()}

rows = []
for label, notes in melodies.items():
    pitches = [n.pitch for n in notes]
    rows.append({
        "rhythm": "normal" if label == "N" else "AFib",
        "notes": len(notes),
        "lowest": note_name(min(pitches)),
        "highest": note_name(max(pitches)),
        "pitch SD (semitones)": round(float(np.std(pitches)), 2),
        "off-scale notes": f"{100 * np.mean([n.chromatic for n in notes]):.0f}%",
    })
pd.DataFrame(rows)

The normal melody never leaves C4 — every beat arrived within 12% of the running mean, so every
note is the root. It is a single repeated pitch, a pedal tone. The AFib melody spans more than
three octaves and leaves the scale for about half its notes.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for ax, (label, notes) in zip(axes, melodies.items()):
    for n in notes:
        ax.hlines(n.pitch, n.start, n.start + n.duration, linewidth=3.5,
                  colors="tab:red" if n.chromatic else "tab:blue")
    name = "Normal rhythm" if label == "N" else "Atrial fibrillation"
    ax.set(title=f"04043 - {name}", ylabel="MIDI pitch", ylim=(20, 95))
    ax.axhline(ROOT_NOTE, color="gray", linewidth=0.6, linestyle=":")
axes[-1].set_xlabel("Time (s)")
axes[0].legend(handles=[Line2D([], [], color="tab:blue", lw=3.5, label="in scale"),
                        Line2D([], [], color="tab:red", lw=3.5, label="chromatic")],
               loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()

---

## 5. Stage 4 — reading an emotion from the music

The music is placed on **Russell's circumplex**, the standard two-axis model of emotion:

- **valence** — unpleasant to pleasant
- **arousal** — calm to energetic

which gives four quadrants: *happy/excited*, *tense/anxious*, *sad/depressed*, *calm/relaxed*.

Three musical cues drive it, chosen from Gabrielsson & Lindström's review of how musical
structure carries emotion:

| Cue | How it is measured | Feeds |
|---|---|---|
| **rhythmic irregularity** | standard deviation of the gaps between note onsets, over their mean | arousal |
| **pitch spread** | standard deviation of the pitches, in semitones | arousal, and (inverted) valence |
| **consonance** | share of notes that stayed in the scale | valence |

**Tempo and loudness are deliberately excluded.** Tempo varies more between people than between
rhythms — in our data every AFib clip also happens to be the faster heart — so including it would
mostly measure whose heart it is. Loudness is re-scaled inside every clip anyway.

The quadrant then picks the instrument the arrangement plays the melody on.

In [ ]:
@dataclass
class Emotion:
    valence: float
    arousal: float
    quadrant: str
    instrument: str
    program: int


def descriptors(notes):
    """The three musical cues the estimate is built from."""
    pitches = np.array([n.pitch for n in notes])
    iois = np.diff([n.start for n in notes])
    irregularity = float(np.std(iois) / np.mean(iois)) if len(iois) and np.mean(iois) > 0 else 0.0
    return {
        "rhythmic_irregularity": float(np.clip(irregularity, 0.0, 1.0)),
        "pitch_spread": float(pitches.std()),
        "consonance": 1.0 - float(np.mean([n.chromatic for n in notes])),
    }


def _normalise(value, bounds):
    low, high = bounds
    return float(np.clip((value - low) / (high - low), 0.0, 1.0))


def valence_arousal(cues):
    """Valence and arousal in [-1, 1]."""
    irregular  = _normalise(cues["rhythmic_irregularity"], IRREGULARITY_RANGE)
    spread     = _normalise(cues["pitch_spread"], SPREAD_RANGE)
    consonance = _normalise(cues["consonance"], CONSONANCE_RANGE)
    arousal = (AROUSAL_WEIGHTS[0] * irregular + AROUSAL_WEIGHTS[1] * spread - 0.5) * 2
    valence = (VALENCE_WEIGHTS[0] * consonance + VALENCE_WEIGHTS[1] * (1 - spread) - 0.5) * 2
    return float(valence), float(arousal)


def estimate_emotion(notes):
    valence, arousal = valence_arousal(descriptors(notes))
    signs = ("+" if valence >= 0 else "-", "+" if arousal >= 0 else "-")
    quadrant = next(name for name, q in QUADRANTS.items()
                    if (q["valence"], q["arousal"]) == signs)
    q = QUADRANTS[quadrant]
    return Emotion(valence, arousal, quadrant, q["instrument"], q["program"])

In [ ]:
rows = []
for label, notes in melodies.items():
    cues = descriptors(notes)
    e = estimate_emotion(notes)
    rows.append({"rhythm": "normal" if label == "N" else "AFib",
                 "irregularity": round(cues["rhythmic_irregularity"], 3),
                 "pitch spread": round(cues["pitch_spread"], 2),
                 "consonance": round(cues["consonance"], 2),
                 "valence": round(e.valence, 2), "arousal": round(e.arousal, 2),
                 "quadrant": e.quadrant, "instrument": e.instrument})
pd.DataFrame(rows)

---

## 6. Stage 3b — the arrangement: harmony and a pulse

The melody alone is sparse. The arrangement adds two parts.

**Strings** play `C – Am – F – G`, one chord every four beats. The chord *sours with the rhythm*:
take the average $\lvert d_i \rvert$ of the four beats under it, and

- below 12% → the chord plays as written, consonant;
- 12% to 25% → it becomes a **diminished triad** `(0, 3, 6)`, which is unstable and wants to
  resolve;
- above 25% → a **semitone cluster** `(0, 1, 6)`, which is simply harsh.

So the harmony is a second, independent reading of the same irregularity: you hear it in the
melody as wrong notes and underneath as a chord that will not settle. For reference, the normal
clips never exceeded 5.5% average deviation, so they never leave the consonant tier.

**A lub-dub drum pair** marks every heartbeat — the first hit on the R-peak, the second a short
delay after it, as loud as that beat's melody note. It keeps the listener aware that every event
is a real heartbeat rather than a musical choice.

Mixed on record 04043, the strings sit 7–9 dB and the pulse 12–17 dB below the melody.

In [ ]:
@dataclass
class Chord:
    start: float
    end: float
    pitches: list
    tier: str
    velocity: int


@dataclass
class Track:
    name: str
    program: int
    notes: list = field(default_factory=list)
    is_drum: bool = False


def tension_tier(mean_deviation):
    if mean_deviation >= CHAOTIC_AT:
        return "chaotic"
    return "tense" if mean_deviation >= TENSE_AT else "consonant"


def harmonize(notes):
    """One chord per BEATS_PER_CHORD beats along C-Am-F-G, soured as the beats get irregular."""
    chords = []
    for i in range(0, len(notes), BEATS_PER_CHORD):
        group = notes[i:i + BEATS_PER_CHORD]
        root, quality = PROGRESSION[(i // BEATS_PER_CHORD) % len(PROGRESSION)]
        tier = tension_tier(np.mean([abs(n.deviation) for n in group]))
        shape = CHORD_SHAPES.get(tier, quality)
        chords.append(Chord(start=group[0].start, end=group[-1].start + group[-1].rr,
                            pitches=[root + interval for interval in shape],
                            tier=tier, velocity=CHORD_VELOCITY[tier]))
    return chords


def heartbeat_pulse(notes):
    """A 'lub-dub' bass-drum pair on every beat, as loud as that beat's melody note."""
    hits = []
    for n in notes:
        lub = int(np.interp(n.velocity, VELOCITY_RANGE, PULSE_VELOCITY_RANGE))
        dub_at = n.start + min(DUB_DELAY_FRACTION * n.rr, DUB_DELAY_MAX)
        hits.append(Note(LUB_DRUM, n.start, DRUM_HIT_SECONDS, lub))
        hits.append(Note(DUB_DRUM, dub_at, DRUM_HIT_SECONDS, int(lub * DUB_VELOCITY_RATIO)))
    return hits


def arrange(notes, melody_program):
    """The three tracks of the piece, plus the chord list (for plots and statistics)."""
    chords = harmonize(notes)
    harmony = [Note(p, c.start, c.end - c.start, c.velocity) for c in chords for p in c.pitches]
    tracks = [
        Track("melody", melody_program, notes),
        Track("harmony", STRINGS_PROGRAM, harmony),
        Track("pulse", DRUM_KIT_PROGRAM, heartbeat_pulse(notes), is_drum=True),
    ]
    return tracks, chords

In [ ]:
for label, notes in melodies.items():
    chords = harmonize(notes)
    tiers = [c.tier for c in chords]
    name = "normal" if label == "N" else "AFib"
    print(f"{name:>6}: {len(chords):>2} chords - "
          f"{tiers.count('consonant')} consonant, {tiers.count('tense')} tense, "
          f"{tiers.count('chaotic')} chaotic")

---

## 7. MIDI, audio and the score

MIDI is written with `pretty_midi`, then rendered to WAV by FluidSynth using a General MIDI
soundfont.

One detail that matters: these pieces are very sparse, and a raw render peaks at only about **5%**
of full scale — almost inaudible on a laptop. Every render is therefore normalised so the loudest
sample sits at 85% of full scale.

In [ ]:
def write_midi(tracks, path):
    midi = pretty_midi.PrettyMIDI()
    for track in tracks:
        instrument = pretty_midi.Instrument(program=track.program, is_drum=track.is_drum,
                                            name=track.name)
        instrument.notes.extend(
            pretty_midi.Note(velocity=int(n.velocity), pitch=int(n.pitch),
                             start=n.start, end=n.start + n.duration)
            for n in track.notes)
        midi.instruments.append(instrument)
    midi.write(str(path))
    return Path(path)


def normalise(wav_path, level=PEAK_LEVEL):
    """Scale the loudest sample to `level` of full scale, in place."""
    rate, data = wavfile.read(str(wav_path))
    full_scale = np.iinfo(data.dtype).max
    peak = np.abs(data.astype(np.int64)).max()
    if peak:
        scaled = data.astype(np.float64) * (level * full_scale / peak)
        wavfile.write(str(wav_path), rate,
                      np.clip(scaled, -full_scale - 1, full_scale).astype(data.dtype))


def render_wav(midi_path, wav_path):
    if shutil.which("fluidsynth") is None:
        raise RuntimeError("FluidSynth is not installed "
                           "(Ubuntu: sudo apt install fluidsynth fluid-soundfont-gm)")
    result = subprocess.run(["fluidsynth", "-ni", SOUNDFONT, str(midi_path),
                             "-F", str(wav_path), "-r", str(SAMPLE_RATE)],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"fluidsynth failed: {result.stderr.strip()}")
    normalise(wav_path)
    return Path(wav_path)


def render(tracks, base_path, audio=True):
    """Write base_path.mid and, if `audio`, base_path.wav."""
    base_path = Path(base_path)
    midi_path = write_midi(tracks, base_path.with_suffix(".mid"))
    wav_path = render_wav(midi_path, base_path.with_suffix(".wav")) if audio else None
    return midi_path, wav_path

### The score plot

Three panels on one shared time axis: the ECG with its detected beats, the beat-to-beat deviation
with the ±12% thresholds marked, and the generated score with the melody coloured by whether it
stayed in the scale and the chords coloured by tension tier.

This single figure shows the whole chain from signal to music, which makes it the most useful
thing to put in front of someone.

In [ ]:
TIER_COLORS = {"consonant": "tab:green", "tense": "tab:orange", "chaotic": "tab:purple"}


def plot_score(clip, notes, chords, title="", path=None):
    fig, (ecg, dev, score) = plt.subplots(3, 1, figsize=(12, 8), sharex=True,
                                          gridspec_kw={"height_ratios": [1, 1, 2]})
    ecg.plot(np.arange(len(clip.signal)) / clip.fs, clip.signal, linewidth=0.6)
    ecg.plot(clip.peaks / clip.fs, clip.signal[clip.peaks], "rx", markersize=4)
    ecg.set(title=title, ylabel="ECG (mV)")

    dev.stem([n.start for n in notes], [n.deviation * 100 for n in notes], basefmt=" ")
    for sign in (1, -1):
        dev.axhline(sign * CHROMATIC_THRESHOLD * 100, color="red", linewidth=0.6, linestyle="--")
    dev.set_ylabel("R-R deviation (%)")

    for c in chords:
        score.hlines(c.pitches, c.start, c.end, colors=TIER_COLORS[c.tier], linewidth=7, alpha=0.35)
    for n in notes:
        score.hlines(n.pitch, n.start, n.start + n.duration, linewidth=3.5,
                     colors="tab:red" if n.chromatic else "tab:blue")
    pitches = [n.pitch for n in notes] + [p for c in chords for p in c.pitches]
    low, high = min(pitches) - 2, max(pitches) + 2
    c_notes = [p for p in range(24, 109, 12) if low <= p <= high]
    score.set_ylim(low, high)
    score.set_yticks(c_notes, [f"C{p // 12 - 1}" for p in c_notes])
    score.set(ylabel="Pitch", xlabel="Time (s)")
    score.legend(handles=[
        Line2D([], [], color="tab:blue", lw=3.5, label="melody, in scale"),
        Line2D([], [], color="tab:red", lw=3.5, label="melody, chromatic"),
        *[Line2D([], [], color=color, lw=7, alpha=0.35, label=f"chord, {tier}")
          for tier, color in TIER_COLORS.items()],
    ], loc="upper center", bbox_to_anchor=(0.5, -0.22), fontsize=8, ncol=5, frameon=False)
    fig.tight_layout()
    if path is not None:
        fig.savefig(path, dpi=130)
    return fig

---

## 8. Putting the stages together

`compose` runs the whole chain on one clip; `save` writes the files.

In [ ]:
@dataclass
class Piece:
    notes: list
    emotion: Emotion
    tracks: list
    chords: list

    def summary(self):
        pitches = [n.pitch for n in self.notes]
        tiers = [c.tier for c in self.chords]
        return {
            "n_notes": len(self.notes),
            "pitch_min": min(pitches), "pitch_max": max(pitches),
            "pitch_std": float(np.std(pitches)),
            "pct_chromatic": 100 * float(np.mean([n.chromatic for n in self.notes])),
            "valence": self.emotion.valence, "arousal": self.emotion.arousal,
            "quadrant": self.emotion.quadrant, "instrument": self.emotion.instrument,
            **{f"{tier}_chords": tiers.count(tier)
               for tier in ("consonant", "tense", "chaotic")},
        }


def compose(clip):
    """Melody, emotion estimate and three-part arrangement for one clip."""
    notes = melody(clip)
    if not notes:
        raise ValueError("the clip is too short to make any notes")
    emotion = estimate_emotion(notes)
    tracks, chords = arrange(notes, emotion.program)
    return Piece(notes, emotion, tracks, chords)


def save(piece, clip, out_dir, title="", audio=True):
    """Write melody.mid, arrangement.mid, score.png and (if `audio`) the WAVs."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    plain = Track("melody", PIANO_PROGRAM, piece.notes)   # NOT the emotion-chosen instrument
    files = {}
    files["melody_mid"], files["melody_wav"] = render([plain], out_dir / "melody", audio)
    files["arrangement_mid"], files["arrangement_wav"] = render(piece.tracks,
                                                                out_dir / "arrangement", audio)
    files["score_png"] = out_dir / "score.png"
    fig = plot_score(clip, piece.notes, piece.chords,
                     f"{title}: {piece.emotion.quadrant}, melody on {piece.emotion.instrument}",
                     files["score_png"])
    plt.close(fig)
    return files

**One detail worth noticing in `save`.** It writes *two* renders. `arrangement` uses the
instrument the emotion model chose, but `melody` is always plain **piano**, whatever the model
said. That second file is what the listening study uses, because if the instrument itself carried
the emotion label the test would be circular — a listener calling a clip "tense" might only be
reacting to a distortion guitar.

In [ ]:
pieces = {label: compose(clip) for label, clip in clips.items()}

for label, piece in pieces.items():
    files = save(piece, clips[label], EXAMPLES_DIR / f"04043_{label}",
                 f"afdb 04043 {label}", audio=RENDER_AUDIO)
    s = piece.summary()
    print(f"{label:>5}: {s['quadrant']:<13} valence {s['valence']:+.2f}  "
          f"arousal {s['arousal']:+.2f}  on {s['instrument']}")
    print(f"       wrote {files['melody_mid'].name}, {files['arrangement_mid'].name}, score.png"
          + ("  (+ WAV)" if RENDER_AUDIO else "  (no audio - set RENDER_AUDIO = True)"))

In [ ]:
fig = plot_score(clips["N"], pieces["N"].notes, pieces["N"].chords,
                 "afdb 04043 - normal rhythm")
plt.show()

In [ ]:
fig = plot_score(clips["AFIB"], pieces["AFIB"].notes, pieces["AFIB"].chords,
                 "afdb 04043 - atrial fibrillation")
plt.show()

Compare the two figures. In the normal clip the middle panel sits flat and well inside the dashed
±12% lines, so the melody is a straight line on C4 over unbroken green chords. In the AFib clip
the deviations cross the thresholds constantly, the melody scatters across three octaves in red,
and the chords turn orange and purple.

---

## 9. The full experiment — 14 pieces from 7 recordings

Set `BUILD_ALL = True` at the top to run this. It streams 7 records from PhysioNet, so expect a
few minutes. With `BUILD_ALL = False` it loads the saved table instead, if one is present.

In [ ]:
def build_examples(records=EXAMPLE_RECORDS, audio=True):
    """Compose every example record and write output/examples/ plus the summary table."""
    summary = []
    for db, record in records:
        try:
            record_clip_set = record_clips(record, db)
        except (OSError, ValueError) as error:
            print(f"skipping {db}/{record}: {error}")
            continue
        for label, clip in record_clip_set.items():
            name = f"{record}_{label}"
            piece = compose(clip)
            save(piece, clip, EXAMPLES_DIR / name, f"{db} {record} {label}", audio)
            row = {"db": db, "record": record, "label": label,
                   "rmssd_ms": round(clip.rmssd_ms, 1), "mean_rr_ms": round(clip.mean_rr_ms)}
            row.update({k: (round(v, 2) if isinstance(v, float) else v)
                        for k, v in piece.summary().items()})
            summary.append(row)
            print(f"{name:>11}: {row['quadrant']:<13} valence {row['valence']:+.2f}  "
                  f"arousal {row['arousal']:+.2f}  chords {row['consonant_chords']}/"
                  f"{row['tense_chords']}/{row['chaotic_chords']}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    frame = pd.DataFrame(summary)
    frame.to_csv(OUTPUT_DIR / "examples_summary.csv", index=False)
    return frame


SUMMARY_CSV = OUTPUT_DIR / "examples_summary.csv"
if BUILD_ALL:
    summary = build_examples(audio=RENDER_AUDIO)
elif SUMMARY_CSV.exists():
    summary = pd.read_csv(SUMMARY_CSV, dtype={"record": str})
    print(f"loaded {len(summary)} rows from {SUMMARY_CSV}")
else:
    summary = None
    print("No results yet. Set BUILD_ALL = True at the top and re-run.")

### Results

The published numbers, for comparison — this is what a correct run reproduces:

| Record | Rhythm | bpm | RMSSD (ms) | Notes | Pitch SD | Off-scale | Chords c/t/x | Valence | Arousal | Emotion |
|---|---|---|---|---|---|---|---|---|---|---|
| afdb 04015 | normal | 80 | 7.6 | 40 | 0.0 | 0% | 10/0/0 | +1.00 | −0.97 | calm |
| afdb 04015 | AFib | 137 | 126.1 | 67 | 9.8 | 57% | 5/9/3 | −0.67 | +0.76 | tense |
| afdb 04043 | normal | 108 | 8.5 | 53 | 0.0 | 0% | 14/0/0 | +1.00 | −0.96 | calm |
| afdb 04043 | AFib | 128 | 134.9 | 63 | 8.5 | 52% | 6/10/0 | −0.51 | +0.45 | tense |
| afdb 04126 | normal | 79 | 18.6 | 39 | 1.3 | 0% | 10/0/0 | +0.93 | −0.79 | calm |
| afdb 04126 | AFib | 160 | 141.9 | 79 | 10.6 | 66% | 3/12/5 | −0.89 | +0.85 | tense |
| afdb 06995 | normal | 83 | 17.8 | 40 | 1.2 | 0% | 10/0/0 | +0.93 | −0.81 | calm |
| afdb 06995 | AFib | 112 | 122.8 | 55 | 8.0 | 51% | 6/7/1 | −0.45 | +0.31 | tense |
| afdb 08455 | normal | 79 | 8.3 | 38 | 0.0 | 0% | 10/0/0 | +1.00 | −0.96 | calm |
| afdb 08455 | AFib | 122 | 99.9 | 59 | 5.9 | 31% | 11/4/0 | +0.07 | +0.02 | happy (borderline) |
| mitdb 201 | normal | 50 | 64.1 | 24 | 1.9 | 0% | 6/0/0 | +0.89 | −0.69 | calm |
| mitdb 201 | AFib | 101 | 194.3 | 50 | 9.8 | 52% | 3/9/1 | −0.58 | +0.76 | tense |
| mitdb 202 | normal | 56 | 37.4 | 27 | 2.0 | 0% | 7/0/0 | +0.89 | −0.65 | calm |
| mitdb 202 | AFib | 125 | 146.9 | 61 | 7.6 | 33% | 9/5/2 | −0.07 | +0.44 | tense |

**What this shows.**

- **Heartbeats.** In every recording the AFib clip is 3–17× as variable as the normal clip.
- **Melody.** Every normal melody stays entirely in the scale with a pitch SD of at most 2
  semitones; three never leave C4. Every AFib melody leaves the scale for 31–66% of its notes.
- **Harmony.** All 67 chords under the normal pieces are consonant. Under the AFib pieces, 68 of
  111 (61%) turn tense or chaotic.
- **Emotion.** All 7 normal pieces read *calm*; 6 of 7 AFib pieces read *tense*. Every AFib piece
  has lower valence and higher arousal than the same person's normal piece — 7 of 7 on both axes.

**The seventh AFib piece, 08455, is the interesting one.** It is a genuinely mild AFib stretch —
fewest off-scale notes, smallest pitch spread, mostly consonant chords — and it lands at the dead
centre of the plane, just inside *happy*. The system is reporting **degree**, not a category. AFib
is not all-or-nothing clinically either.

A caution: these are 7 recordings from **6 people**. PhysioNet's header says `mitdb` record 202
came from the same analog tape as record 201 — both are a 68-year-old man taking propranolol and
digoxin, which is why his "normal" clips run at 50–56 bpm.

In [ ]:
if summary is not None:
    normal = summary[summary["label"] == "N"].set_index("record")
    afib   = summary[summary["label"] == "AFIB"].set_index("record")
    shared = normal.index.intersection(afib.index)

    comparison = pd.DataFrame({
        "RMSSD normal":  normal.loc[shared, "rmssd_ms"],
        "RMSSD AFib":    afib.loc[shared, "rmssd_ms"],
        "x more variable": (afib.loc[shared, "rmssd_ms"] / normal.loc[shared, "rmssd_ms"]).round(1),
        "off-scale normal": normal.loc[shared, "pct_chromatic"].round(0),
        "off-scale AFib":   afib.loc[shared, "pct_chromatic"].round(0),
        "d valence": (afib.loc[shared, "valence"] - normal.loc[shared, "valence"]).round(2),
        "d arousal": (afib.loc[shared, "arousal"] - normal.loc[shared, "arousal"]).round(2),
    })
    print(f"AFib has lower valence in {(comparison['d valence'] < 0).sum()}/{len(shared)} recordings")
    print(f"AFib has higher arousal in {(comparison['d arousal'] > 0).sum()}/{len(shared)} recordings")
    display(comparison)

In [ ]:
def plot_circumplex(rows, ax=None):
    """Every piece on Russell's plane; an arrow runs from each record's normal to its AFib."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 7))
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, color="lightgray"))
    ax.axhline(0, color="gray", linewidth=0.6)
    ax.axvline(0, color="gray", linewidth=0.6)
    for x, y, text in ((0.6, 1.05, "happy / excited"), (-0.6, 1.05, "tense / anxious"),
                       (-0.6, -1.1, "sad / depressed"), (0.6, -1.1, "calm / relaxed")):
        ax.text(x, y, text, ha="center", color="dimgray", fontsize=10)

    by_record = {}
    for row in rows:
        by_record.setdefault(str(row["record"]), {})[row["label"]] = (float(row["valence"]),
                                                                     float(row["arousal"]))
    for record, points in by_record.items():
        if {"N", "AFIB"} <= points.keys():
            ax.annotate("", xy=points["AFIB"], xytext=points["N"],
                        arrowprops={"arrowstyle": "->", "color": "lightgray"})
        for label, (valence, arousal) in points.items():
            is_normal = label == "N"
            ax.scatter(valence, arousal, marker="o" if is_normal else "^", s=60,
                       color="tab:blue" if is_normal else "tab:red", zorder=3)
            ax.annotate(record, (valence, arousal), textcoords="offset points",
                        xytext=(5, 4), fontsize=8)

    ax.set(xlim=(-1.25, 1.25), ylim=(-1.25, 1.25), aspect="equal",
           xlabel="valence (unpleasant -> pleasant)", ylabel="arousal (calm -> energetic)",
           title="Every example on Russell's plane (arrow: normal -> AFib)")
    ax.legend(handles=[Line2D([], [], marker="o", color="tab:blue", ls="", label="normal rhythm"),
                       Line2D([], [], marker="^", color="tab:red", ls="", label="AFib")],
              loc="lower left", fontsize=8)
    return ax


if summary is not None:
    plot_circumplex(summary.to_dict("records"))
    plt.tight_layout()
    plt.show()

Every arrow points the same way: down and to the left in valence, up in arousal. The blue circles
cluster tightly in the bottom-right *calm* quadrant; the red triangles scatter across the top-left
*tense* quadrant — except 08455, sitting at the origin.

---

## 10. Record 219 — where the system disagreed with the database, and was right

`mitdb` record 219 also has both rhythms, but it is **excluded** from the results above. Here is
why, and why it then became one of the two most interesting findings in the project.

Even its steadiest window labelled *normal* has an **RMSSD of 270.6 ms** — nearly twice its AFib
clip at 145.9 ms. The annotations explain it: **133 non-conducted P-waves**, atrial beats blocked
before they reach the ventricles. That is a genuine irregularity which the label "normal" does not
capture.

Every stage of the pipeline flagged the record independently, without being told anything:

- the emotion model reads its *normal* clip as **tense** and its AFib clip as **calm**;
- the harmony under the normal clip has tense and chaotic chords (3/2/2), while the AFib clip
  stays mostly consonant (9/1/0).

The pipeline is being consistent: it reflects the actual rhythm, not the label. Here the rhythm
behind the "normal" label is not normal.

That matters for the claim this project can make. It is not "we detect atrial fibrillation" —
algorithms already do that better. It is that **the music is an instrument for noticing things
about the rhythm that the label does not carry.**

In [ ]:
control = record_clips("219", "mitdb")
control_pieces = {label: compose(clip) for label, clip in control.items()}

rows = []
for label, piece in control_pieces.items():
    s = piece.summary()
    rows.append({"label on the database": "normal" if label == "N" else "AFib",
                 "RMSSD (ms)": round(control[label].rmssd_ms, 1),
                 "off-scale": f"{s['pct_chromatic']:.0f}%",
                 "chords c/t/x": f"{s['consonant_chords']}/{s['tense_chords']}/{s['chaotic_chords']}",
                 "our verdict": s["quadrant"]})
display(pd.DataFrame(rows))
print("\nThe clip the database calls normal is the more irregular of the two.")

---

## 11. The listening study — the one test that is still missing

Everything above is a measurement on our own output. **No human has been asked whether any of it
is audible.** For a sonification that is the gap that matters, and it is the cheapest one to
close.

The design:

- **16 clips** — the 14 published, plus both record 219 clips as a hidden control.
- **The plain melody**, on piano, never the emotion-chosen instrument. Otherwise the test is
  circular.
- Converted to mono mp3 and renamed `clip_01`…`clip_16`, in one fixed random order with no two
  clips from the same record adjacent.
- Four questions per clip: pleasantness (1–9), energy (1–9), a forced choice between
  calm/tense/happy/sad, and perceived rhythmic steadiness (1–9).

Why the fourth question exists: *can they hear the irregularity* and *does it change how they
feel* are two different claims, and without separating them we could not tell which step of the
pipeline worked.

Listeners are told nothing about hearts until they finish.

In [ ]:
SEED = 20261008


def presentation_order(sources, seed=SEED):
    """Shuffle so no two clips of one record are adjacent and no rhythm runs four deep."""
    rng = random.Random(seed)
    for _ in range(10000):
        trial = sources[:]
        rng.shuffle(trial)
        if any(trial[i][0] == trial[i - 1][0] for i in range(1, len(trial))):
            continue
        if any(len({trial[j][1] for j in range(i - 3, i + 1)}) == 1
               for i in range(3, len(trial))):
            continue
        return trial
    raise RuntimeError("no ordering satisfied the spacing constraints")


def build_study(out_dir=STUDY_DIR, seed=SEED):
    """Blind the rendered melodies into clip_01..clip_16 and write the key beside them."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    available = [(d.name.split("_")[0], d.name.split("_", 1)[1])
                 for d in sorted(EXAMPLES_DIR.iterdir())
                 if d.is_dir() and (d / "melody.wav").exists()]
    if not available:
        raise SystemExit("no rendered melodies - set RENDER_AUDIO and BUILD_ALL to True")
    ordered = sorted(available, key=lambda pair: (pair[0], 0 if pair[1] == "N" else 1))

    key = []
    for position, (record, label) in enumerate(presentation_order(ordered, seed), 1):
        clip_id = f"clip_{position:02d}"
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error",
                        "-i", str(EXAMPLES_DIR / f"{record}_{label}" / "melody.wav"),
                        "-ac", "1", "-codec:a", "libmp3lame", "-b:a", "96k",
                        str(out_dir / f"{clip_id}.mp3")], check=True)
        key.append({"clip": clip_id, "position": position, "record": record, "label": label,
                    "rhythm": "normal" if label == "N" else "afib",
                    "excluded_record": record == "219"})
        print(f"{clip_id} <- {record}_{label}")
    pd.DataFrame(key).to_csv(out_dir / "KEY.csv", index=False)
    print(f"\n{len(key)} clips in {out_dir}. KEY.csv maps them back - never show it to listeners.")
    return key


if BUILD_STUDY:
    study_key = build_study()
else:
    print("BUILD_STUDY is False - set it to True (with RENDER_AUDIO and BUILD_ALL) to build the set.")

### Analysing the ratings

Four questions, four answers. Each one can come out against us, and each outcome means something:

| Outcome | What it would mean |
|---|---|
| Listeners track the model on both axes | The translation works end to end. The sonification carries clinically meaningful structure to an untrained ear. |
| They hear the irregularity but their emotion ratings don't move | The mapping is *audible* but not *affective*. We would have located the break precisely — a sharper finding than a flat success. |
| They hear nothing reliable | The mapping is measurable but not perceptible. A real negative result about sonification design. |
| Only trained musicians separate the clips | The information is there but needs a trained ear. |

There is no version of this study that returns nothing.

In [ ]:
WORD_TO_QUADRANT = {"calm": "calm_relaxed", "tense": "tense_anxious",
                    "happy": "happy_excited", "sad": "sad_depressed"}


def analyse_ratings(ratings, key, model):
    """ratings: rows of (listener, clip, pleasant, energy, word, steady).

    key:   clip -> {record, label, rhythm, excluded_record}
    model: (record, label) -> {valence, arousal, quadrant}
    """
    frame = pd.DataFrame(ratings)
    frame["record"] = frame["clip"].map(lambda c: key[c]["record"])
    frame["rhythm"] = frame["clip"].map(lambda c: key[c]["rhythm"])

    per_clip = frame.groupby("clip")[["pleasant", "energy", "steady"]].mean()
    per_clip["listener_valence"] = (per_clip["pleasant"] - 5) / 4   # 1-9 -> -1..+1
    per_clip["listener_arousal"] = (per_clip["energy"] - 5) / 4
    per_clip["model_valence"] = [model[(key[c]["record"], key[c]["label"])]["valence"]
                                 for c in per_clip.index]
    per_clip["model_arousal"] = [model[(key[c]["record"], key[c]["label"])]["arousal"]
                                 for c in per_clip.index]

    print("1. Does listener rating track the model? (Spearman across clips)")
    for axis in ("valence", "arousal"):
        rho, p = stats.spearmanr(per_clip[f"model_{axis}"], per_clip[f"listener_{axis}"])
        print(f"   {axis:<8} rho = {rho:+.3f}   p = {p:.4f}")

    print("\n2. AFib minus normal, per recording")
    diffs = []
    for record, group in frame.groupby("record"):
        if key[group['clip'].iloc[0]]["excluded_record"]:
            continue
        means = group.groupby("rhythm")[["pleasant", "energy", "steady"]].mean()
        if {"normal", "afib"} <= set(means.index):
            diffs.append({"record": record,
                          **(means.loc["afib"] - means.loc["normal"]).round(2).to_dict()})
    diffs = pd.DataFrame(diffs)
    display(diffs)
    for column, expected in (("pleasant", "lower"), ("energy", "higher"), ("steady", "lower")):
        if len(diffs) >= 5:
            _, p = stats.wilcoxon(diffs[column])
            print(f"   {column:<9} median {diffs[column].median():+.2f}  p = {p:.4f}  "
                  f"(expected {expected} for AFib)")

    print("\n3. Forced choice vs the model's quadrant")
    hits = sum(WORD_TO_QUADRANT.get(r["word"]) ==
               model[(key[r["clip"]]["record"], key[r["clip"]]["label"])]["quadrant"]
               for r in ratings)
    result = stats.binomtest(hits, len(ratings), 0.25, alternative="greater")
    print(f"   {hits}/{len(ratings)} = {100 * hits / len(ratings):.1f}% (chance 25%)  "
          f"p = {result.pvalue:.3g}")

    print("\n4. Record 219 - the hidden control")
    control = frame[frame["clip"].map(lambda c: key[c]["excluded_record"])]
    if not control.empty:
        by_rhythm = control.groupby("rhythm")["steady"].mean()
        display(by_rhythm.round(2))
        if by_rhythm.get("normal", 9) < by_rhythm.get("afib", 0):
            print("   Listeners heard the NORMAL-labelled clip as the less steady one -")
            print("   the same inversion the pipeline found, confirmed independently by ear.")
    return per_clip


print("analyse_ratings is ready - call it when the responses are in.")

---

## 12. What this shows, and what it does not

Being clear about this is the most important part of the notebook.

### Shown

1. The sonification is **faithful and reproducible** — the same rules, independently
   reimplemented, give identical numbers on all 14 clips.
2. The music **separates the two rhythms in every recording**, on note content, harmony and the
   emotion estimate alike.
3. The system conveys **degree, not just category** (record 08455).
4. It **caught a mislabelled record** (219), flagged independently by every stage.

### Not shown

1. **That any listener can hear it.** Every claim above is a measurement on our own output.
   That is what §11 is for.
2. **That the music's emotion reflects the person's emotion.** This is the project's central
   hypothesis and it is untested. The databases used here contain no emotional or psychological
   data at all, so it *cannot* be tested on them.
3. **That the mapping is right for healthy hearts.** Slow, relaxed breathing *raises*
   beat-to-beat variability — respiratory sinus arrhythmia — while acute stress lowers it. Our
   rule would call a calm, healthy person tense. This is the clearest known defect in the system.
4. **That this beats plain HRV.** The emotion model is, in effect, a musical re-description of
   heart-rate variability. Whether routing through music adds information is open.

### The honest statement

> The system detects how calm or agitated **the heart's rhythm sounds**, and names that with the
> vocabulary of music emotion. Whether it corresponds to the person's inner emotion is the
> hypothesis, not the finding.

### Also worth knowing

Two pretrained music-emotion models — Essentia and music2emo — were run on these pieces as
independent second opinions. All three methods hear a difference in every recording, but **no two
agree on both axes**: Essentia hears the AFib pieces as *higher* in valence in 7 of 7 recordings,
music2emo as *lower* in 6 of 7. Part of the reason is tempo, which the pretrained models hear and
ours deliberately ignores. The rest is that none of these models has ever heard music like this.

That disagreement is itself a finding: pretrained music-emotion models are unreliable on sparse,
synthetic, out-of-distribution music.

### Ethics

The example recordings are from adult cardiac patients at Beth Israel Hospital, Boston, 1975–1983,
under Open Data Commons Attribution 1.0. Running this on anyone's own ECG produces personal health
data, and any use with a specific group — the original motivation involved autistic people — would
need its own validation and ethical review. Calling someone's heart rhythm "tense" is a claim
about them, and on the evidence here it is not one we can make.

---

## 13. References

**Data**
- MIT-BIH Atrial Fibrillation Database — https://physionet.org/content/afdb/
- MIT-BIH Arrhythmia Database — https://physionet.org/content/mitdb/
- Goldberger et al. (2000). PhysioBank, PhysioToolkit, and PhysioNet. *Circulation* 101(23).

**Emotion**
- Russell (1980). A circumplex model of affect. *Journal of Personality and Social Psychology*.
- Gabrielsson & Lindström (2010). The role of structure in the musical expression of emotions.

**For validating Stage 4** (ECG recorded with self-reported emotion)
- WESAD — https://archive.ics.uci.edu/dataset/465/wesad+wearable+stress+and+affect+detection
- DREAMER — https://zenodo.org/records/546113
- AMIGOS — https://www.eecs.qmul.ac.uk/mmv/datasets/amigos/index.html

**Tools** — wfdb, pretty_midi, FluidSynth, Essentia, music2emo.